# Results for model: gemma3:27b

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np

def solve():
    train_df = pl.read_parquet('./jane-street-real-time-market-data-forecasting/train.parquet')

    def calculate_top_quantile(df):
        df = df.sort("feature_00", descending=True)
        quantile_threshold = df["feature_00"].quantile(0.15)
        df = df.with_columns(pl.when(df["feature_00"] >= quantile_threshold).then(1).otherwise(0).alias("top_quantile"))
        return df

    def process_batch(batch):
        batch = batch.with_columns(
            pl.col("feature_00").rank().over("date_id", descending=True).alias("feature_00_rank")
        )
        batch = batch.with_columns(
            pl.when(batch["feature_00_rank"] <= int(len(batch) * 0.15))
            .then(1)
            .otherwise(0)
            .alias("top_quantile")
        )
        return batch

    train_df = train_df.group_by("date_id").map(process_batch)

    X = train_df.select(['top_quantile']).to_numpy()
    y = train_df.select(['responder_6']).to_numpy().flatten()

    model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, seed=42)
    model.fit(X, y)

    return model

if __name__ == '__main__':
    model = solve()
    print("Model trained successfully.")
    # Example prediction (optional)
    # test_df = pl.read_parquet('./jane-street-real-time-market-data-forecasting/test.parquet')
    # test_df = test_df.with_columns(pl.col("feature_00").rank().over("date_id", descending=True).alias("feature_00_rank"))
    # test_df = test_df.with_columns(
    #     pl.when(test_df["feature_00_rank"] <= int(len(test_df) * 0.15))
    #     .then(1)
    #     .otherwise(0)
    #     .alias("top_quantile")
    # )
    # X_test = test_df.select(['top_quantile']).to_numpy()
    # predictions = model.predict(X_test)
    # print(predictions)
    pass
